In [ ]:
!pip install tensorflow
!pip install scikeras
#https://github.com/ml-jku/lsc/blob/master/pythonCode/lstm/models.py
from joblib import dump, Parallel, delayed
from functools import partial
from tensorflow.keras.layers import LSTM, Conv1D, Embedding, Dropout, AveragePooling1D, GlobalMaxPooling1D, GlobalAveragePooling1D, MaxPooling1D, AveragePooling1D, TimeDistributed
from tensorflow.keras.layers import Input
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import tensorflow as tf
from scikeras.wrappers import KerasRegressor
from tensorflow.keras.optimizers import Adam, SGD
from sklearn.model_selection import GridSearchCV
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
import pandas as pd
import os
import joblib
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, make_scorer
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
from tensorflow.keras.layers import Reshape
import numpy as np
from tensorflow.keras import regularizers
from tensorflow.keras.layers import Flatten, Bidirectional
import random as rn
from keras.preprocessing.text import Tokenizer
from google.colab import drive


In [ ]:
#allow acess to my google drive where the files are
drive.mount('/content/drive')
google_drive_path = "/content/drive/MyDrive"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def later_padding(ohe_matrix, smiles_maxlen, total_unique_char):
  """
  Add horizontal and vertical padding
  to the given matrix using numpy.pad() function.

  Parameters
  ----------
  ohe_matrix : ndarray
      Character array.
  smiles_maxlen : int
      Maximum length of the SMILES string.
  total_unique_char : int
      Total number of unique characters in the SMILES dataset.

  Returns
  -------
  padded_matrix : numpy.ndarray
          Padded one-hot encoded matrix of
          shape (unique char in smiles, max smile_length).
  """
  row_padding = smiles_maxlen - len(ohe_matrix)
  #print('this the the ohe matrix', ohe_matrix)
  #print('row_padding', row_padding)
  col_padding = (total_unique_char+1) - ohe_matrix.shape[1]
  #print('col_padding', col_padding)

  # If the padding sizes are negative, make them 0
  row_padding = max(0, row_padding)
  col_padding = max(0, col_padding)
  padded_matrix = np.pad(ohe_matrix, ((0, row_padding), (0, col_padding)), "constant")

  return padded_matrix
def preprocessor(smiles, islaterpadding, smiles_maxlen, num_words):
  """
  One-hot encodes the SMILES using keras
  implementation.

  Parameters
  ----------
  canonical_char : array
      Canonical character array.
  islaterpadding : bool
      The paramater is `True` if later_padding is required,
      `False` otherwise.
  smiles_maxlen : int
      Maximum length of the SMILES string.
  unique_char : list
      List of unique characters in the string data set.

  Returns
  -------
  encoded : numpy.ndarray
      One-hot encoded matrix of shape
      (chars in SMILES, length of SMILES).
  """

  # Integer encoding using Tokenizer
  encoded_list = []
  for input_smiles in smiles:
    integer_encoded = tokenizer.texts_to_sequences(input_smiles)
    #print('this is the integer_encoded', integer_encoded)
    # One-hot encoding using to_categorical function
    encoded = to_categorical(integer_encoded)
    #print('this is encoded', encoded)
    if islaterpadding:
      encoded = later_padding(encoded, smiles_maxlen, num_words)
      #print('this is encoding after padding', encoded)
    encoded = encoded.transpose()
    encoded_list.append(encoded)


  #print('this is encoding after padding_2', encoded)
  return encoded_list

In [ ]:
seed = 126
kfold = KFold(n_splits=10, shuffle=True, random_state=seed)
#build the function
def create_model(seq_length, data_dim):
  model = Sequential()
  model.add(Dropout(0.5, input_shape=(seq_length, data_dim)))

  # Conv1D layer
  model.add(Conv1D(filters=16, kernel_size=4, strides=2, kernel_initializer='lecun_uniform', activation='selu', padding='same', use_bias=False))
  model.add(Dropout(0.2))
  model.add(Conv1D(filters=16, kernel_size=4, strides=2, kernel_initializer='lecun_uniform', activation='selu', padding='same', use_bias=False))
  model.add(Dropout(0.2))
  model.add(AveragePooling1D())

  #LSTM layers

  model.add(LSTM(units=128, go_backwards=True, return_sequences= True, recurrent_activation='sigmoid'))
  model.add(Dropout(0.1))
  model.add(LSTM(units=512, go_backwards=False, return_sequences=False, recurrent_activation='sigmoid'))
  model.add(Dropout(0.1))

  model.add(Dense(1, activation='linear', kernel_initializer='lecun_uniform',
                  bias_initializer='zeros', kernel_regularizer=regularizers.l1_l2(0, 1e-9)))

  # Compile the model
  optimizer = Adam(learning_rate=0.01)

  model.compile(loss='mean_absolute_error', optimizer=optimizer, metrics = ['mean_absolute_error'])

  return model

def calculate_metrics(model, X, y, kfold):
  r2_scores = []
  mse_scores = []
  mae_scores = []

  for train_index, test_index in kfold.split(X):
    # Use list comprehension to handle 3D arrays
    print([X[i].shape for i in train_index])  # Print shapes of training samples
    print([X[i].shape for i in test_index])
    X_train = np.stack([X[i] for i in train_index])
    X_test = np.stack([X[i] for i in test_index])

    y_train = y[train_index]
    y_test = y[test_index]
    model.fit(X_train, y_train, epochs=20, batch_size=batch_size)
    print(model)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2_scores.append(r2)
    mse_scores.append(mse)
    mae_scores.append(mae)
  r2_mean = np.mean(r2_scores)
  r2_std = np.std(r2_scores)
  mse_mean = np.mean(mse_scores)
  mse_std = np.std(mse_scores)
  mae_mean = np.mean(mae_scores)
  mae_std = np.std(mae_scores)

  return r2_mean, r2_std, mse_mean, mse_std, mae_mean, mae_std

In [ ]:
#load the input data
df = pd.read_csv("/content/drive/My Drive/Colab Notebooks/input_data_with_canonical_smiles.csv", delimiter=",")
print(df)
print(list(df.columns))
targets = df['target'].unique()
print(targets)
print(len(targets))

#since the GPU time would end every 4 h, I save the targets already seen on a txt file. The next time GPU was available at google colab, the targets already seen would be removed from the full list of targets
with open("/content/drive/My Drive/Colab Notebooks/input_targets_seen_seed126.txt", 'r', encoding='utf-8-sig') as file:
  targets_seen = [int(line) for line in file.read().splitlines()]
targets_seen = list(set(targets_seen))
print(targets_seen)
print(len(targets_seen))

# Get the list of targets
targets_diff = df['target'].unique()
print(targets_diff)
print(len(targets_diff))

targets = [target for target in targets_diff if target not in targets_seen]
print(targets)
print(len(targets))


self._target(*self._args, **self._kwargs)
  File "<ipython-input-5-5d21d5fefe10>", line 13, in move_mouse
  File "/usr/local/lib/python3.10/dist-packages/mouse/__init__.py", line 107, in wheel
    _os_mouse.wheel(delta)
  File "/usr/local/lib/python3.10/dist-packages/mouse/_nixmouse.py", line 123, in wheel
    build_device()
  File "/usr/local/lib/python3.10/dist-packages/mouse/_nixmouse.py", line 73, in build_device
    device = aggregate_devices('mouse')
  File "/usr/local/lib/python3.10/dist-packages/mouse/_nixcommon.py", line 141, in aggregate_devices
    uinput = make_uinput()
  File "/usr/local/lib/python3.10/dist-packages/mouse/_nixcommon.py", line 29, in make_uinput
    fcntl.ioctl(uinput, UI_SET_EVBIT, EV_KEY)
OSError: [Errno 25] Inappropriate ioctl for device


                                                   smiles   target  activity  \
0       C[C@H]1C[C@H](O)[C@H](O)[C@@H](O)[C@H]1N[C@H]1...     2074  7.167491   
1       CNC(=O)C1=C(C)NC(C)=C(C(=O)NCCCN2CCC(c3ccccc3)...      232  6.356968   
2             N#C/C(=C\c1ccc(O)c(O)c1)C(=O)c1ccc(O)c(O)c1     1824  4.757364   
3       CC(=O)N[C@@H]1[C@H](NC(=O)[C@H](C)NC(=O)[C@@H]...     3890  2.301030   
4       O=C(O)CC1(CS[C@H](CCc2cccc(C3(O)CCC3)c2)c2cccc...     1798  9.327902   
...                                                   ...      ...       ...   
758114  COC/C=C/C1=CC2=CC(=O)C(C)(OC(=O)c3ccco3)C(=O)C...  1293240  5.226223   
758115        Cc1cc(C)cc(NC(=O)c2cncc(-c3ccc(Cl)cc3)n2)c1     5451  8.522879   
758116       Cc1cc(C)cc(NC(=O)c2cncc(-c3ccc(C#N)cc3)n2)c1     5451  8.096910   
758117     CC(C)Oc1cccc(NC(=O)c2cncc(-c3ccc(C#N)cc3)n2)c1     5451  7.309804   
758118  C=CC(=O)Nc1cccc(-n2c(=O)n(C)c3cnc(Nc4ccc(OCCOC...     5251  8.386158   

                                       

In [ ]:
# Loop over the targets
for target in targets:
  # Select data for current target
  data = df[df['target'] == target]
  X_smiles = data['canonical_smiles']
  print(X_smiles)
  print(X_smiles.shape)
  tokenizer = Tokenizer(char_level=True, filters='', lower=False)  # char_level=True means that each character will be treated as a token
  tokenizer.fit_on_texts(X_smiles.values)
  print('this is the index', tokenizer.word_index)
  accepted_chars = set("".join(list(X_smiles)))
  print('accepted_chars is', accepted_chars)
  print('len accepted chars is', len(max(accepted_chars)))
  num_words = len(accepted_chars)
  print('number of words 2', num_words)
  print(f'accepter characters are: {accepted_chars}')
  print('We have', num_words, 'different words.')

  # maximum sentence length for zero padding
  max_length_target = [len(smiles) for smiles in X_smiles]
  print("Max SMILES length:", max_length_target)
  sentence_length = np.max(max_length_target)
  print('Maximum sentence length is:', sentence_length)

  print('max lenght', sentence_length)
  X = preprocessor(smiles=X_smiles, islaterpadding=True, smiles_maxlen=sentence_length, num_words=num_words)
  print(f'the target is {target}')
  y = data["activity"]
  y = y.values.reshape(-1, 1)
  print(y)
  print(y.shape)
  #define hyperparameter grid

  batch_size=int(len(X_smiles)*0.2)
  print(f'the lenght of X is {len(X_smiles)} which gives a batch_size of {batch_size}')

  #define model
  model = KerasRegressor(model=create_model, seq_length=num_words +1,
                          data_dim=sentence_length, batch_size=batch_size,
                          epochs=20, verbose=0, random_state=seed)
  r2_mean, r2_std, mse_mean, mse_std, mae_mean, mae_std = calculate_metrics(model, X, y, kfold)
  with open(f'{google_drive_path}/metrics_seed{seed}.txt', 'a') as f:
    f.write('target\tr2\tstd r2\tmse\tstd mse\tmae\tstd mae\n')
  with open(f'{google_drive_path}/metrics_seed{seed}.txt', 'a') as f:
    f.write(f'{target}\t{r2_mean}\t{r2_std}\t{mse_mean}\t{mse_std}\t{mae_mean}\t{mae_std}\n')
  with open(f'{google_drive_path}/targets_seen_seed{seed}.txt', 'a') as f:
    f.write(f'{target}\n')


10356      CCCCc1oc2ccccc2c1C(=O)c1cc(I)c(OCCN(CC)CC)c(I)c1
88843     CN(C)C(=O)CCC(=O)Nc1cc(F)cc(-c2noc([C@H]3CCCCN...
88846     COc1ccc(Cl)cc1CN1C(=O)C2(CC2)c2ccc(C(=O)Nc3ccc...
88852      Cc1ccc(-c2nc([C@H]3CCCCN3C(=O)COc3ccccc3)cs2)cn1
114147    COc1ccc(Cl)cc1CN1C(=O)OC(C)(C)c2ccc(C(=O)Nc3nc...
                                ...                        
730405    COc1ccc(-c2nc([C@H]3CCCCN3C(=O)COc3ccccc3)cs2)cn1
730406    CCN(c1cc(C(=O)Nc2ccn(CC(=O)O)n2)cc2c1OCO2)S(=O...
743662    COCCN(c1cc(C(=O)Nc2ccc(CC(=O)O)cc2)cc2c1OCO2)S...
743663    COc1ccc(Cl)cc1S(=O)(=O)N(CCC(=O)O)c1cc(C(=O)Nc...
743720    COc1ccc(Cl)cc1S(=O)(=O)N(c1cc(C(=O)Nc2ccc(CC(=...
Name: canonical_smiles, Length: 485, dtype: object
(485,)
this is the index {'c': 1, 'C': 2, '(': 3, ')': 4, 'O': 5, '1': 6, '=': 7, '2': 8, '3': 9, 'N': 10, 'n': 11, '[': 12, ']': 13, '@': 14, 'H': 15, 'l': 16, 'S': 17, '-': 18, '4': 19, 'o': 20, 'F': 21, 's': 22, '+': 23, '#': 24, 'B': 25, '.': 26, 'P': 27, 'I': 28, 'r': 29, 'a': 30, '/

[(33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146), (33, 146)

A saída de streaming foi truncada nas últimas 5000 linhas.
 [5.49485002]
 [5.49620932]
 [5.31875876]
 [8.40893539]
 [8.10237291]
 [6.        ]
 [6.        ]
 [7.        ]
 [8.03151705]
 [7.65490196]
 [7.4202164 ]
 [6.69897   ]
 [6.79588002]
 [7.76955108]
 [5.62590599]
 [7.85387196]
 [6.55284197]
 [7.92081875]
 [6.25181197]
 [6.60205999]
 [8.19382003]
 [8.34678749]
 [8.20760831]
 [8.49485002]
 [7.92081875]
 [8.49485002]
 [8.11350927]
 [7.1739252 ]
 [6.85387196]
 [7.69897   ]
 [6.29242982]
 [5.20065945]
 [7.        ]
 [7.        ]
 [7.        ]
 [6.        ]
 [6.        ]
 [6.        ]
 [6.        ]
 [7.        ]
 [6.        ]
 [7.        ]
 [5.94882432]
 [4.64793331]
 [6.56863624]
 [8.09691001]
 [5.95860731]
 [7.38721614]
 [6.15490196]
 [6.88605665]
 [6.88605665]
 [7.27572413]
 [7.        ]
 [6.49485002]
 [5.4202164 ]
 [7.1739252 ]
 [7.        ]
 [7.        ]
 [6.        ]
 [7.85387196]
 [6.33724217]
 [4.69897   ]
 [6.00877392]
 [7.18708664]
 [7.26760624]
 [6.76955108]
 [5.38721614]
 [6